Creating a Chatbot that will answer user question based on the details supplied to the AI. It will act as Abhinaw on Abhinaw's website and answer all user question. If it doesn't know an answer it will sent a push notification to Abhinaw about the question. It will also take user details so that Abhinaw can get back with the answer. It will also evaluate the response generated by 1st LLM.

In [1]:
from dotenv import load_dotenv
import os
import docx2txt
from openai import OpenAI
import gradio as gr
import json
import requests

In [2]:
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
gemini_key = os.getenv("GOOGLE_API_KEY")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

openai_client = OpenAI()
gemini_client = OpenAI(api_key=gemini_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")


Creating a Push Function that will use Pushover tool and send notification to my phone.

In [3]:
def push(message):
    #print(f"Pushover message. {message}")
    payload = {"user":pushover_user,"token":pushover_token,"message":message}
    requests.post(pushover_url, data=payload)

Creating Tool Function for sending Push notification based on user input.

In [4]:
def push_user_detail(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [5]:
def push_user_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

Tool Function JSON, TO be used i AI message

In [6]:
push_user_detail_json = {"name": "push_user_detail",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [7]:
push_user_question_json = {"name": "push_user_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }}


In [8]:
tools = [{"type": "function", "function": push_user_detail_json},
        {"type": "function", "function": push_user_question_json}]


Creating function to handle tool call

In [9]:
def handle_tool_call(tool_calls):
    results = []
    for tool_call in tool_calls:
        func_name = tool_call.function.name
        func_args  = json.loads(tool_call.function.arguments)
        print(f"Tool called: {func_name}", flush=True)

        if func_name == "push_user_detail":
            result = push_user_detail(**func_args)
        elif func_name == "push_user_question":
            result = push_user_question(**func_args)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [10]:
ResumeDoc = docx2txt.process("D:\Desktop\Abhinaw_Kumar.docx")
name = "Abhinaw"

with open(r"C:\Users\coola\projects\agents\1_foundations\me\summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\coola\AppData\Local\Temp\ipykernel_7652\2002845344.py:1: SyntaxWarning: invalid escape sequence '\D'
  ResumeDoc = docx2txt.process("D:\Desktop\Abhinaw_Kumar.docx")


In [18]:
system_prompt = f"""You are acting as {name}. You have to answer user querry on {name}'s website pretending as {name}. You have to answer user query based on the details\
    provided in details shared in {name}'s resume and {name}'s summary. The details are present in {ResumeDoc} and {summary}.\
    You have to act professionally and be accurate to the details provided to the best of your knowledge. \
    You have to behave very politely and professionaly as representing {name}.\
    If you don't know the answer to any question, use your push_user_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
    If the user wants to interact with {name} for any query and provide his email-id then use your push_user_detail tool to record user detail and inform {name}.\
    Also if you are not able to answer a user question then after capturing the details, inform the user that {name} will get back to the user at the earliest. Be polite and professional in your reply."""

system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

Adding a validation model for the reply generated by 1st Model.

In [11]:
evaluator_promt = f"""You are acting as the evaluator for the response generated by 1st Agent.\
The Agent is working as {name} and is generating the response while interacting with user on {name}'s website. \
Resposne is generated based on details present in {ResumeDoc}, {summary} and user input. You need to evaluate the resposne for the quality of the response and correctness based on the user input and details provided to the model.\
If the 1st agent wants to use it's tool then accept the request and let the model use its tool based on the requirement. Using a tool for question whose answer is not know should be an acceptable behaviour.\
You have to be very professional while evaluating the response. You should act with utmost responsibility and have to validate details for {name}."""

evaluator_promt += f"With the above context, work very professionaly while evaluating the repsonse."

In [12]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [13]:
def evaluator_user_input(reply, history, message):
    user_input  = f"Input message from the user is \n\n {message}\n\n"
    user_input += f"History of the chat \n\n {history}\n\n"
    user_input += f"Response from the agent. \n\n {reply}"
    user_input += f"Please evaluate the response, replying with whether it is acceptable and your feedback."
    
    return user_input

In [14]:
def re_run(reply,history,message,feedback):
    print(f"Inside Re-run because Evaluation failed with feedback: {feedback}")
    rerun_prompt = system_prompt + f"Your Response \n\n {reply}\n\n for user input \n\n{message}\n\n failed the evaluation by another agent."
    rerun_prompt += f"The reason of failure of your response is \n\n{feedback}\n\n"
    rerun_prompt += f"History of the chat \n\n{history}\n\n"

    messages = [{"role":"system", "content":rerun_prompt}] +history+ [{"role":"user", "content":message}]
    
    response = openai_client.chat.completions.create(model = "gpt-5.4-mini",messages=messages)
    return response.choice[0].message.content
       


In [15]:
def evaluate(reply, history, message) -> Evaluation:
    print("Inside the evaluate Function")
    messages = [
        {"role":"system","content":evaluator_promt},
        {"role":"user", "content": evaluator_user_input(reply, history, message)}
    ]
        
    response = openai_client.responses.parse(model = "gpt-5.4-mini", input = messages, text_format=Evaluation)
    reply_text: Evaluation = response.output_parsed
    print(reply_text)

    return reply_text

CHAT1 function in which LLM will generate a message based on user input and details from input file. If the detail is not available for the user input, it will use tool and if answer is present than the reply is evaluated by another LLM.

In [16]:
def CHAT1(message,history):
    messages = [{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    
    done = False
    while not done:
        print(f"Calling LLM for user input {message}")
        response = gemini_client.chat.completions.create(model="gemini-2.5-flash",messages = messages, tools= tools)
        #response = openai_client.chat.completions.create(model="gpt-5.4-mini",messages = messages, tools= tools)
        reply = response.choices[0].message.content
        
        finish_reason = response.choices[0].finish_reason
        print(f"This task is finished and reason is {finish_reason}")
        
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            print("Going to call Tool")
            tool_calls = message.tool_calls
            results = handle_tool_call(tool_calls)
            messages.append(message)
            messages.extend(results)
            
        else:
            done = True
            print(f"Before going for Evaluate reply generated is {reply}")
            eval_resp = evaluate(reply, history, message)
            print(f"Response received from Evaluate = {eval_resp}")
            
            if not eval_resp.is_acceptable:
                print("Failed evaluation - retrying")
                print(eval_resp.feedback)
                reply = re_run(reply, history, message, eval_resp.feedback)
            
                
    return reply

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## 💬 Chatbot for My Work Experience")
    chat = gr.ChatInterface(CHAT1, type="messages")
    clear_btn = gr.Button("Clear Chat")
    clear_btn.click(fn=lambda: None, inputs=None, outputs=chat)

demo.launch()

